<a href="https://colab.research.google.com/github/Yashika-Bayeen/agentic-ai/blob/main/Day_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FAISS (Facebook AI Similarity Search)
FAISS is an open-source library developed by Facebook AI for efficient similarity search and clustering of dense vectors.

In [ ]:
# Install libraries
!pip install faiss-cpu sentence-transformers rank-bm25 numpy groq python-dotenv -q
print("✅ Libraries installed.")

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load tiny encoder
model = SentenceTransformer('all-MiniLM-L6-v2')

CORPUS = [
    "FAISS Flat index performs brute-force L2 search.",
    "IVF index divides vectors into multiple Voronoi cells.",
    "HNSW index uses multi-layer proximity graphs for logarithmic search.",
    "Hybrid search merges keyword search with vector database retrieval."
]

# Embed documents
corpus_embeddings = model.encode(CORPUS).astype('float32')
dimension = corpus_embeddings.shape[1]

# 1. FLAT index
index_flat = faiss.IndexFlatL2(dimension)
index_flat.add(corpus_embeddings)

# 2. HNSW index
index_hnsw = faiss.IndexHNSWFlat(dimension, 32)
index_hnsw.add(corpus_embeddings)

# Search Flat index
query = "brute force vector search"
q_emb = model.encode([query]).astype('float32')
distances, indices = index_flat.search(q_emb, k=1)

print("FAISS Flat Index Best Match:")
print(f"  Index: {indices[0][0]} | Distance: {distances[0][0]:.4f}")
print(f"  Document: '{CORPUS[indices[0][0]]}'")

This cell initializes the `SentenceTransformer` model, defines a corpus of documents, and creates two types of FAISS indexes: `IndexFlatL2` and `IndexHNSWFlat`. It then demonstrates a basic search using the `IndexFlatL2`.

In [ ]:
def evaluate_recall(queries, ground_truths, index, embed_model, corpus, k=2):
    hits = 0
    for q, gt in zip(queries, ground_truths):
        q_vec = embed_model.encode([q]).astype('float32')
        _, indices = index.search(q_vec, k=k)
        retrieved_docs = [corpus[idx] for idx in indices[0] if idx != -1]

        if gt in retrieved_docs:
            hits += 1

    return hits / len(queries)

test_queries = ["tell me about voronoi cells in indexing", "what is L2 search in FAISS"]
test_gts = [CORPUS[1], CORPUS[0]]

recall_val = evaluate_recall(test_queries, test_gts, index_flat, model, CORPUS, k=2)
print(f"Recall@2 Accuracy: {recall_val * 100:.1f}%")

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize corpus for BM25
tokenized_corpus = [doc.lower().split(" ") for doc in CORPUS]
bm25 = BM25Okapi(tokenized_corpus)

def hybrid_search_rrf(query_str, k_constant=60, top_n=2):
    # 1. Sparse search ranking
    query_tokens = query_str.lower().split(" ")
    sparse_scores = bm25.get_scores(query_tokens)
    sparse_ranking = np.argsort(sparse_scores)[::-1]

    # 2. Dense search ranking
    q_vec = model.encode([query_str]).astype('float32')
    _, dense_indices = index_flat.search(q_vec, len(CORPUS))
    dense_ranking = dense_indices[0]

    # 3. Apply RRF scoring formula: Score = sum( 1 / (k + rank) )
    rrf_scores = {}
    for rank, doc_idx in enumerate(sparse_ranking):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + 1.0 / (k_constant + rank + 1)

    for rank, doc_idx in enumerate(dense_ranking):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + 1.0 / (k_constant + rank + 1)

    # Sort indices by score
    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return [(idx, CORPUS[idx], rrf_scores[idx]) for idx in sorted_indices[:top_n]]

hybrid_results = hybrid_search_rrf("L2 vector distance Flat search")
print("Hybrid RRF Search Results:")
for r in hybrid_results:
    print(f"  Index: {r[0]} | Score: {r[2]:.6f} | '{r[1]}'")

In [ ]:
from sentence_transformers import CrossEncoder

# Load lightweight cross-encoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def search_and_rerank(query_str, top_k_initial=3):
    # Step 1: Retrieve candidates using FAISS
    q_vec = model.encode([query_str]).astype('float32')
    _, indices = index_flat.search(q_vec, top_k_initial)
    candidates = [CORPUS[idx] for idx in indices[0]]

    # Step 2: Formulate pairs [Query, Candidate]
    pairs = [[query_str, doc] for doc in candidates]

    # Step 3: Grade similarity with Cross-Encoder
    scores = reranker.predict(pairs)

    # Step 4: Re-rank candidates by cross-encoder score
    ranked_candidates = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return ranked_candidates

reranked = search_and_rerank("proximity graph hierarchical networks")
print("Reranked results (higher score is better):")
for doc, score in reranked:
    print(f"  Score: {score:.4f} | Document: '{doc}'")

In [ ]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def generate_synthetic_qa(document_text):
    prompt = (
        "Analyze the document below and generate exactly 2 short questions whose answers are found in the text. "
        "Provide output in raw JSON format: {\"qa_pairs\": [{\"question\": \"...\", \"answer\": \"...\"}]}\n\n"
        f"[DOCUMENT]\n{document_text}"
    )

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

synthetic_set = generate_synthetic_qa(CORPUS[2])
print("Generated Synthetic Q&A Pairs:")
print(json.dumps(synthetic_set, indent=2))